# Fucrimodo as a Library

Fucrimodo provides a library for writing the config files for the
****fucrimodo lab**** or integrating it into other programs. This
notebook tutorial shows how to define a very basic multi-stage GA.
Please read the \`Getting started\` section of the documentation before
you start.

## Random Numbers

Since many aspects of GAs depend on stochastic processes, we need to
pass a pseudo-random number generator to all parts of the algorithm to
ensure reproducible behaviour. Here we set an rng.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)


## Target File Definition

Target files are the main input of a fucrimodo run. They store the
target SOAP descriptor, the descriptor type, and the parameters needed
to calculate it.

We create a temporary directory to store the file in and later the
results. You can change this to any other directory if you want to store
results elsewhere. (Note: please delete the \`temp<sub>dir</sub>\`
yourself once you no longer need it; otherwise it will be deleted
automatically when you restart your computer.)

In [ ]:
import tempfile
from pathlib import Path
tmp_dir = Path(tempfile.mkdtemp(prefix="fucrimodo_example_"))
print("Directory created at:", tmp_dir)


### Generate the Target File

In a real fucrimodo application the target structure is unknown.
However, for testing we can create a target file from an example
compound. The \`Individual\` class inherits from ASE \`Atoms\`, so it
can be initialized in the same way as an ASE \`Atoms\` structure. Please
refer to the docs for more information. From this mock individual we can
then calculate the descriptor.

In [ ]:
from fucrimodo.core import Individual

# For this test case we generate a very simple example individual
target_structure = Individual(
     numbers=[26],
     positions=[[0, 0, 0]],
     cell = [
         [3, 0, 0],
         [0, 3, 0],
         [0, 0, 3],
     ],
     pbc=True,
)

# Now we can calculate the descriptor from the target structure
from fucrimodo.customs.global_soap_target import GlobalSOAP
soap_kwargs = {
    "r_cut": 15.0,
    "n_max": 8,
    "l_max": 8,
    "sigma": 0.5,
    "species": list(target_structure.get_chemical_symbols()),
    "periodic": True,
    "average": "inner"
}

soap = GlobalSOAP(**soap_kwargs)
target_features = soap.create(target_structure)

print("Generated target descriptor with shape:", target_features.shape)

del target_structure
print("Deleted target structure, to show it is not needed for the inversion.")


The \`utils\` module provides a simple function that makes storing the
target file very easy:

In [ ]:
from fucrimodo.utils import target_file_parser

target_file_path = tmp_dir / "target_file.json"
target_file_parser.save_to_target_file(
    features = list(target_features),
    descriptor_name = "GlobalSOAP",
    descriptor_parameters = soap_kwargs,
    save_path = target_file_path,
    additional_notes = f"Number of atoms: 1",
)

print("Created targetfile at", target_file_path)


### Load the Target File

From here on we can pretend the target structure never existed. We now
parse the target file without any knowledge of the target structure.
Again, fucrimodo provides a simple method for this.

In [ ]:
soap_obj, target_features, _ = target_file_parser.load_target_file(target_file_path)


Note: If a descriptor other than GlobalSOAP is used, please load the
target file manually, because currently only the GlobalSOAP descriptor
is supported. Use the source code of the existing method as a starting
point.

## Generate the Initial Population

The algorithm needs an initial population to start. A population is
basically a storage container of the individuals at the current state of
the algorithm. Fucrimodo includes start population generators to create
an initial population. Here we use a random sampling approach guided by
knowledge of the target descriptor. Look in the documentation for more
details on the generator.

### Set Up a Population Generator

We first need to set up some hyperparameters:

In [ ]:
import fucrimodo.core.utils as core_utils

# Defines the smallest allowed distance between
# two neighbouring atoms based on their covalent radii.
closest_distances = core_utils.CustomClosestDistances(
    species=soap_obj.species, ratio_of_covalent_radii=0.7
)

# Minimum and maximum cell sizes of generated structures
cell_bound = core_utils.CustomCellBounds(
    {
        "a": [1, 100],
        "b": [1, 100],
        "c": [1, 100],
        "alpha": [10, 170],
        "beta": [10, 170],
        "gamma": [10, 170],
    }
)

# RBF similarity is the most important metric for analysing how closely
# each candidate resembles the target descriptor.
from fucrimodo.customs.fitness_functions import SoapRbfSimilarityFitness
similarity_fitnesses = SoapRbfSimilarityFitness(
    target_features,
    soap_object = soap_obj,
    db_title = "small_rbf_sim_fitness",
    rbf_gamma = 0.01,
)

# Set an estimated number of atoms as a hyperparameter.
# If unknown, repeat the algorithm with different values.
n_atoms = 1

# Please specify the number of physical cores that can be used for
# multiprocessing.
n_cores = 16

# Generate a population using estimates based on the SOAP descriptor
from fucrimodo.customs.population_generators import RandomSampleCrystalPopulation
population_generator = RandomSampleCrystalPopulation(
    soap_obj=soap_obj,
    target_features=target_features,
    closest_distances=closest_distances,
    fitness_functions=similarity_fitnesses,
    n_atoms=n_atoms,
    n_jobs=n_cores,
    rng=rng,
)




### Generate the Population

The individuals in the initial population can now be investigated.

In [ ]:
population = population_generator.generate_population(50)
print(f"Generated population with {len(population)} individuals.")

from ase.visualize.plot import plot_atoms
plot_atoms(population.individuals[0], rotation=('45x, 45y, 0z'))

# Uncomment to interactively inspect the generated structures
# from ase.visualize import view
# view(population.individuals)


## Set Up a Multi-Stage Search

### Initialize the MultiStageSearch Object

To run a multi-stage GA we first need to set up the
\`\`MultiStageSearch\`\` object. This class is used to manage data,
time-keeping, and organization of stages during a multi-stage search.
Please refer to the documentation for more information.

In [ ]:
from fucrimodo.core.multi_stage_search import MultiStageSearch
multi_stage_search = MultiStageSearch(
    save_dir=tmp_dir,
    target_features=target_features,
    descriptor_object=soap_obj,
    descriptive_name="example_run",
)

print("Data will be stored at:", multi_stage_search.run_dir)


### Set Up Global Statistics

To track statistics for the run we can add a global statistics
dictionary to the \`\`MultiStageSearch\`\` object. It is a dictionary
whose keys are the names of the statistics and whose values are any
methods that accept an individual as input and return a number.

In [ ]:
# We can even use lambda functions to define simple statistics.
global_statistics_dict = {
    "Volume": lambda ind: ind.get_volume()
}

# More complex statistics can also be added.
# Here we use the similarity of a candidate's descriptor to the target.
rbf_similarity_fitness = SoapRbfSimilarityFitness(
    target_soap_features=target_features,
    soap_object=soap_obj,
    rbf_gamma=0.1,
)
global_statistics_dict["RBF_Similarity"] = rbf_similarity_fitness.evaluate_individual

# Now we add the global statistics dict to the MultiStageSearch object.
multi_stage_search.global_statistics_dict = global_statistics_dict

print("Following statistics will be tracked:", [s for s in multi_stage_search.global_statistics])


### Define the Exploration GA Stage

We can now set up the stages. Each stage is an independent optimisation
algorithm. For more information please refer to the documentation.

The only stage type currently implemented in fucrimodo is the GA stage.
A GA optimises a population of individuals (here atomic structures)
based on evolutionary principles. During each iteration (called a
generation) the algorithm combines the properties of selected
individuals to form a new population using the crossover operator. The
properties of individuals are also randomly changed through the mutation
operator. The individuals are then evaluated with a fitness function.
Finally, individuals are selected based on their fitness values and
passed on to the next generation. This process is repeated until a
breaking condition is reached.

Fucrimodo implements a novel kind of GA algorithm: the multi-stage GA.
In it, different stages of the GA follow different goals. This tutorial
introduces an exploration stage and an optimisation stage.

We first define the exploration stage. It uses strong mutation and
crossover operators that allow it to explore novel structures in the
search space.

In [ ]:
from fucrimodo.customs import population_selections, break_conditions, fitness_functions
from fucrimodo.customs.ga_stage import (
    GAStage,
    crossovers,
    mutations,
)

name = "explore_ga"
description = "This stage explores the search space for structure."

# Stop when the maximum number of generations is reached
# or an individual with sufficiently high fitness is found.
break_condition = break_conditions.MultipleOrBreak(
    [
        break_conditions.GenerationBreak(100),
        break_conditions.MaxFitnessBreak(0, 0.99)
    ]
)

# Use the similarity fitness since it is the main optimisation objective.
# Also use a physicality fitness that penalizes individuals in which atoms are
# closer than the defined threshold.
fitness_func_list =  [
    rbf_similarity_fitness,
    fitness_functions.PhysicalityFitness(
        closest_distances = core_utils.CustomClosestDistances(
            species=soap_obj.species, ratio_of_covalent_radii=0.9
        )
    ),
]

# Mutations during exploration should change the structures strongly
# so that new parts of the search space can be explored.
mutation_list = [
    mutations.pos_mut.RattleMutation(
        closest_distances=closest_distances,
        rattle_strength=0.5,
        rattle_prop=0.8,
        rng=rng,
    ),
    mutations.cell_mut.StrainMutation(
        closest_distances=closest_distances,
        n_variable_cell_vectors=3,
        stddev=0.1,
        rng=rng,
    ),
    mutations.cell_mut.StrainMutation(
        closest_distances=closest_distances,
        n_variable_cell_vectors=1,
        stddev=0.3,
        rng=rng,
    ),
    mutations.sym_mut.GetConventionalCellMutation(
        closest_distances=closest_distances,
        symmetry_tol=0.3,
        rng=rng,
    ),
    mutations.cell_mut.MinimizeTiltMutation(
        closest_distances=closest_distances,
        rng=rng
    ),
]
mutation_list.append(
    mutations.multi_mut.MultipleMutations(
        mutation_list,
        closest_distances,
        2,
        rng=rng
    )
)

# The crossovers are set up to handle crossing
# between very different structures.
crossover_list = [
    crossovers.CutAndSpliceCrossover(
        closest_distances=closest_distances,
        cell_bounds=cell_bound,
        rng=rng
    ),
    crossovers.CutAndSpliceCrossover(
        closest_distances=closest_distances,
        cell_bounds=cell_bound,
        number_of_variable_cell_vectors=3,
        rng=rng,
    ),
    crossovers.UnitCellCrossover(
        closest_distances=closest_distances,
        rng=rng
    ),
]

# Parent selection chooses the individuals to which the crossover and
# mutation operators are applied.
# In tournament selection, the best of 5 randomly sampled individuals is chosen.
parent_selection = population_selections.TournamentSelection(5, rng=rng)

# Survivor selection chooses which individuals of the population will be
# passed on to the next generation.
# NSGA-II selection chooses individuals that are fit and diverse.
survivor_selection = population_selections.NSGA2Selection()

# All the operators can now be used to create the first GA stage.
explore_ga_stage = GAStage(
    name=name,
    fitness_functions=fitness_func_list,
    crossover_list=crossover_list,
    mutation_list=mutation_list,
    mutation_probability=0.9,
    crossover_probability=0.9,
    break_condition=break_condition,
    parent_selection=parent_selection,
    survivor_selection=survivor_selection,
    parent_ratio=0.5,
    description=description,
    save_n_structures=10,
    rng=rng
)


For more details on the parameters please refer to the documentation.

### Define the Optimization GA Stage

We also create a second GA stage. This stage should optimise the
structures found. This is achieved by using less drastic mutations and
applying more crossovers.

In [ ]:
name = "optimize_ga"
description = "This stage optimizes the found structure."

fitness_func_list =  [rbf_similarity_fitness]

mutation_list = [
    mutations.pos_mut.RattleMutation(
        closest_distances=closest_distances,
        rattle_strength=0.1,
        rattle_prop=0.8,
        rng=rng,
    ),
    mutations.pos_mut.RattleMutation(
        closest_distances=closest_distances,
        rattle_strength=0.2,
        rattle_prop=0.8,
        rng=rng,
    ),
    mutations.cell_mut.StrainMutation(
        closest_distances=closest_distances,
        n_variable_cell_vectors=3,
        stddev=0.1,
        rng=rng,
    ),
    mutations.cell_mut.StrainMutation(
        closest_distances=closest_distances,
        n_variable_cell_vectors=1,
        stddev=0.2,
        rng=rng,
    ),
    mutations.sym_mut.GetConventionalCellMutation(
        closest_distances=closest_distances,
        symmetry_tol=0.3,
        rng=rng,
    ),
    mutations.cell_mut.MinimizeTiltMutation(
        closest_distances=closest_distances,
        rng=rng,
    ),
]
mutation_list.append(
    mutations.multi_mut.MultipleMutations(
        mutation_list,
        closest_distances,
        2,
        rng=rng,
    )
)

crossover_list = [
    crossovers.CutAndSpliceCrossover(
        closest_distances=closest_distances,
        cell_bounds=cell_bound,
        rng=rng,
    ),
    crossovers.CutAndSpliceCrossover(
        closest_distances=closest_distances,
        cell_bounds=cell_bound,
        number_of_variable_cell_vectors=3,
        rng=rng,
    ),
    crossovers.UnitCellCrossover(
        closest_distances=closest_distances,
        rng=rng,
    ),
]

break_condition = break_conditions.MultipleOrBreak(
    [
        break_conditions.GenerationBreak(500),
        break_conditions.MaxFitnessBreak(0, 0.99),
    ]
)

parent_selection = population_selections.TournamentSelection(3, rng=rng,)

survivor_selection = population_selections.NSGA2Selection()

optimize_ga_stage = GAStage(
    name=name,
    fitness_functions=fitness_func_list,
    crossover_list=crossover_list,
    mutation_list=mutation_list,
    mutation_probability=0.5,
    crossover_probability=0.8,
    break_condition=break_condition,
    parent_selection=parent_selection,
    survivor_selection=survivor_selection,
    parent_ratio=0.5,
    description=description,
    save_n_structures=10,
    rng=rng,
)


## Run the Multi-Stage Search

Now everything is set up and we can run the algorithm. To run a stage,
it must be started with the \`\`MultiStageSearch.run\`\` method. We
first run the exploration GA stage.

In [ ]:
multi_stage_search.run(population=population, stage=explore_ga_stage)


After this first stage the optimisation is not yet complete, but we can
analyse the created individuals:

In [ ]:
print("Current generation:", population.generation)
print("N individuals:", len(population))

all_fitness_values = [ind.fitness.values[0] for ind in population.individuals]
print("Max fitness:", max(all_fitness_values))
print("Min fitness:", min(all_fitness_values))

# Uncomment to view the population.
# from ase.visualize import view
# view(population.individuals)



Now the optimisation stage can be run.

In [ ]:
multi_stage_search.run(population=population, stage=optimize_ga_stage)


Again we can analyse the results and confirm that an individual with
high similarity to the target has been found.

In [ ]:
print("Current generation:", population.generation)
all_fitness_values = [ind.fitness.values[0] for ind in population.individuals]
print("Max fitness:", max(all_fitness_values))
print("Min fitness:", min(all_fitness_values))


## Analyse the Run

Fucrimodo also provides utilities for analysing multi-stage GA runs. All
data collected during the run is stored in the run directory.

In [ ]:
multi_stage_search.run_dir


### Retrieve Structures

Analysis of the generated structures can be done with tools from the ASE
library.

For each generation, the 10 best structures are stored in the run
directory inside an ASE database called \`structures.db\`. Structures
can be analysed in Python or with the CLI/web interface provided by ASE.
More information can be found in the [ASE
documentation](https://docs.ase-lib.org/ase/db/db.html).

In [ ]:
from ase.db import connect

db_path = Path(multi_stage_search.run_dir) / "structures.db"
structure_db = connect(db_path)

print(f"Database contains {structure_db.count()} structures.")

# e.g. get the first structure from the database
first_structure = structure_db.get_atoms(id = 11, add_additional_information = True)
print("Structure:\t", first_structure)
print("RBF Similarity: ", first_structure.info["key_value_pairs"]["RBF_Similarity"])


### Analyse Run Metrics

Next we can analyse metrics. \`\`Fucrimodo\`\` comes with helper classes
to organize the results. The run directory of a finished run can be
loaded into the \`\`RunData\`\` class.

In [ ]:
from fucrimodo.analysis.run_analysis import RunData
run_data = RunData(multi_stage_search.run_dir)

print("Total number of generations:", run_data.n_generations)
print("Number of stages performed:", run_data.n_stages)


We can also display the maximum and minimum values of each tracked
global statistic.

In [ ]:
from fucrimodo.analysis.utils import get_statistics_overview
print("Global Metric Statistics:")
print(get_statistics_overview(run_data.global_statistics))


Finally, we can also plot one of the statistics.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()

# Row of the statistic to plot. See overview above.
# Here index 0 is the volume and index 1 the RBF similarity.
row_idx = 1

results_df = run_data.global_statistics.loc[row_idx, "results"]
results_df.plot(
    ax=ax,
    x="gen",
    y=["min", "max", "avg"],
)
ax.set_xlabel("Generation")
ax.set_ylabel("Similarity")

# We can easily plot vertical lines to show where the stages change.
stage_ids = results_df["stage_id"].to_numpy()
change_idx = np.flatnonzero(stage_ids[1:] != stage_ids[:-1])
ax.vlines(change_idx, -100, 100, "black", zorder=0)

ax.set_xlim(0, run_data.n_generations)
ax.set_ylim(min(results_df["min"]), max(results_df["max"]))


## Analyse Stages

\[TODO: Missing\]